In [69]:
from dotenv import load_dotenv
from anthropic import Anthropic

#load environment variable
load_dotenv()

#automatically looks for an "ANTHROPIC_API_KEY" environment variable
client = Anthropic()

model = "claude-haiku-4-5-20251001"

In [70]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system 

    message = client.messages.create(**params)
    return message.content[0].text

In [48]:
messages = []

# Add user input to the list of messages 
add_user_message(messages, "Is tea or coffee better at breakfast?")   

answer = chat(messages)
answer 

"That's a matter of personal preference! Here are some points for each:\n\n**Coffee**\n- Higher caffeine content for a stronger energy boost\n- Rich, bold flavor many people enjoy in the morning\n- May help with focus and alertness\n- Popular breakfast pairing worldwide\n\n**Tea**\n- Gentler caffeine lift, often feels smoother\n- Wide variety of flavors (green, black, herbal, etc.)\n- Can be easier on the stomach for some people\n- Contains L-theanine, which may promote calm focus\n\n**It depends on:**\n- Your caffeine sensitivity\n- Personal taste preferences\n- Cultural traditions\n- Health considerations\n\nMany people swear by their morning coffee, while others find tea a more gentle and enjoyable start to the day. Both can be great breakfast companions!\n\nWhich do you prefer?"

In [ ]:
messages = []

# Add user input to the list of messages 
add_user_message(messages, "Is tea or coffee better at breakfast?")   

add_assistant_message(messages, "Coffee is better because")
# Assistant prefill requires a model that supports it (claude-sonnet-4-6 does not)
answer = chat(messages)

answer

' it has more caffeine, which provides a stronger morning boost to help you wake up and start your day with energy. Tea, while containing some caffeine, is generally milder and may not be as effective at jumpstarting your alertness.\n\nActually, let me reconsider—the "better" choice really depends on what works for you:\n\n**Coffee** if you want:\n- Strong caffeine kick\n- Bold flavor\n- Quick energy boost\n\n**Tea** if you want:\n- Gentler caffeine (steady energy without jitters)\n- Antioxidants and other health benefits\n- Easier on your stomach\n- Calming elements (like L-theanine in green tea)\n\n**Personal factors that matter:**\n- Your caffeine sensitivity\n- Stomach sensitivity\n- Taste preference\n- How much sleep you got\n- Your overall health goals\n\nNeither is objectively "better"—it\'s about what fits your body and preferences.'

## Stop Sequences

In [71]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature, 
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system 

    message = client.messages.create(**params)
    return message.content[0].text

In [61]:
messages = []

# Add user input to the list of messages 
add_user_message(messages, "Count from 1 to 10")   
answer = chat(messages)

answer

'Here is counting from 1 to 10:\n\n1, 2, 3, 4, 5, 6, 7, 8, 9, 10'

In [63]:
messages = []

# Add user input to the list of messages 
add_user_message(messages, "Count from 1 to 10")   
answer = chat(messages, stop_sequences=[", 5"])

answer

'Here is counting from 1 to 10:\n\n1, 2, 3, 4'

## Structured Data

In [ ]:
messages = []

# Add user input to the list of messages 
add_user_message(messages, "Generate a very short event bridge rule as json")  
chat(messages)

'Here\'s a simple EventBridge rule in JSON:\n\n```json\n{\n  "Name": "my-rule",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "my-lambda-target",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:my-function"\n    }\n  ]\n}\n```\n\nThis rule:\n- **Triggers** when an EC2 instance enters the `running` state\n- **Targets** a Lambda function'

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")  
add_assistant_message(messages, "```json") #prefilling can also take language hints for better formatting in some cases

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [74]:
import json 

json.loads(text.strip())

{'Name': 'MySimpleRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}